In [30]:
!pip install pandas chromadb sentence-transformers spacy pdfplumber scikit-learn datasets
!python -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 26.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [31]:
!pip install chromadb

In [34]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
import re

class VectorDBBuilder:
    def __init__(self, db_path="./chroma_db", collection_name="mooc_courses"):
        # Initialize the embedding model (BAAI/bge-base-en-v1.5)
        print("Loading Embedding Model...")
        self.embedding_model = SentenceTransformer('BAAI/bge-base-en-v1.5')

        # Initialize ChromaDB persistent client
        self.client = chromadb.PersistentClient(path=db_path)
        self.collection = self.client.get_or_create_collection(name=collection_name)

    def clean_text(self, text):
        """Standardize text, remove HTML, and extra whitespace."""
        if not isinstance(text, str):
            return ""
        text = re.sub(r'<[^>]+>', '', text) # Remove HTML tags
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def process_and_store(self, csv_path):
        """Loads course data, embeds descriptions/skills, and stores in ChromaDB."""
        print(f"Loading data from {csv_path}...")
        df = pd.read_csv(csv_path)

        # Assuming your CSV has 'course_id', 'title', 'description', 'skills_taught'
        df['clean_description'] = df['description'].apply(self.clean_text)
        df['clean_skills'] = df['skills_taught'].apply(self.clean_text)

        # Create the document string to embed
        df['content_to_embed'] = df['title'] + ". " + df['clean_description'] + " Skills: " + df['clean_skills']

        documents = df['content_to_embed'].tolist()
        metadatas = df[['title', 'clean_skills']].to_dict(orient='records')
        ids = df['course_id'].astype(str).tolist()

        print(f"Generating embeddings for {len(documents)} courses...")
        # BGE models highly recommend adding a prompt for retrieval tasks, but for indexing, raw text is fine.
        embeddings = self.embedding_model.encode(documents, normalize_embeddings=True, show_progress_bar=True)

        print("Storing in ChromaDB...")
        # Batch upload to ChromaDB
        self.collection.add(
            ids=ids,
            embeddings=embeddings.tolist(),
            documents=documents,
            metadatas=metadatas
        )
        print("Vector database successfully populated.")

if __name__ == "__main__":
    # Example usage:
    builder = VectorDBBuilder()
    NERDataPrep
    builder.process_and_store("mooc_dataset.csv")
    pass

Loading Embedding Model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading data from mooc_dataset.csv...
Generating embeddings for 5 courses...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Storing in ChromaDB...
Vector database successfully populated.


In [35]:
from datasets import load_dataset
import re
import json

class NERDataPrep:
    def __init__(self):
        # A seed list of skills to search for in the raw text to generate training data
        self.hard_skills = [
            "Python", "AWS", "Amazon Web Services", "React", "JavaScript",
            "Docker", "Kubernetes", "SQL", "Machine Learning", "Pandas"
        ]

    def find_entity_offsets(self, text, entity_list, label):
        """Finds the start and end character indexes of skills in the text."""
        entities = []
        for entity in entity_list:
            # Use regex to find exact word matches and ignore case
            for match in re.finditer(r'\b' + re.escape(entity) + r'\b', text, flags=re.IGNORECASE):
                start, end = match.span()
                entities.append((start, end, label))

        # Handle overlapping entities (spaCy throws an error if entities overlap)
        # Sort by start index, then by length (descending) to keep the longest match
        entities = sorted(entities, key=lambda x: (x[0], -(x[1]-x[0])))
        cleaned_entities = []
        last_end = -1
        for start, end, lbl in entities:
            if start >= last_end:
                cleaned_entities.append((start, end, lbl))
                last_end = end

        return entities

    def format_huggingface_data(self, dataset_name="jacob-hugging-face/job-descriptions", num_samples=500):
        print(f"Downloading dataset: {dataset_name}...")
        # Load the dataset from Hugging Face
        dataset = load_dataset(dataset_name, split='train')

        training_data = []
        print(f"Formatting {num_samples} samples for spaCy...")

        for i in range(min(num_samples, len(dataset))):
            # The column name might vary ('job_description', 'text', etc.). Adjust if needed.
            text = dataset[i].get('job_description', '')

            if not text:
                continue

            # Keep descriptions reasonably short for training efficiency
            text = text[:1500]

            entities = self.find_entity_offsets(text, self.hard_skills, "HARD_SKILL")

            # Only add to training data if we actually found skills
            if entities:
                training_data.append((text, {"entities": entities}))

        return training_data

if __name__ == "__main__":
    prep = NERDataPrep()
    spacy_training_data = prep.format_huggingface_data()

    print(f"\nSuccessfully generated {len(spacy_training_data)} annotated training examples.")
    print("Sample output:")
    print(spacy_training_data[0])

    # Save to JSON for the training script to use later
    with open("ner_training_data.json", "w") as f:
         json.dump(spacy_training_data, f)
    print("\nSaved to ner_training_data.json")

Formatting 500 samples for spaCy...

Successfully generated 23 annotated training examples.
Sample output:
('description\n\nweb designers looking to expand your professional reach welcome to robert half marketing  creative start the process with robert half today\n\nwe are searching for highly skilled web designers with experience working within corporate brand standards and guidelines the ideal candidates would have advanced skills in creating wireframes designing mobile applications landing pages interactive sites qa testing experience working with various interfaces and familiarity with uxui design principles candidates are expected to have strong skills in adobe photoshop illustrator and indesign any experience in html css and javascript is a major plus familiarity with content management systems is highly preferred\n\nthere is nothing more satisfying when looking for freelance and fulltime creative opportunities than working with someone who knows your area of expertise as industr

In [36]:
import spacy
from spacy.training.example import Example
import random

def train_custom_ner(training_data, output_dir="./custom_ner_model", iterations=30):
    """
    Trains a custom NER model on top of en_core_web_md.
    training_data format: [("Text about React and Python", {"entities": [(15, 20, "HARD_SKILL"), ...]})]
    """
    print("Loading base spacy model...")
    nlp = spacy.load("en_core_web_md")

    # Check if NER is in the pipeline, if not add it
    if "ner" not in nlp.pipe_names:
        ner = nlp.add_pipe("ner", last=True)
    else:
        ner = nlp.get_pipe("ner")

    # Add our custom label
    ner.add_label("HARD_SKILL")
    ner.add_label("SOFT_SKILL")
    ner.add_label("TOOL")

    # Disable other pipes during training to only train NER
    pipe_exceptions = ["ner", "trf_wordpiecer", "trf_tok2vec"]
    unaffected_pipes = [pipe for pipe in nlp.pipe_names if pipe not in pipe_exceptions]

    print("Beginning training loop...")
    with nlp.disable_pipes(*unaffected_pipes):
        optimizer = nlp.resume_training()
        for itn in range(iterations):
            random.shuffle(training_data)
            losses = {}
            for text, annotations in training_data:
                doc = nlp.make_doc(text)
                example = Example.from_dict(doc, annotations)
                nlp.update([example], drop=0.35, sgd=optimizer, losses=losses)
            print(f"Iteration {itn + 1} - Losses: {losses}")

    print(f"Saving custom model to {output_dir}...")
    nlp.to_disk(output_dir)

# Note: You will need to parse the Hugging Face job descriptions dataset
# to generate the `training_data` list before running this.

In [37]:
%%writefile main.py
import argparse
import json
import os

Overwriting main.py


In [38]:
!python main.py --setup

In [39]:
!python main.py --analyze --resume sample_resume.pdf --jd sample_jd.txt

In [44]:
%%writefile skill_gap_analyzer.py
import spacy
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import chromadb
import numpy as np
import pdfplumber

class SkillGapAnalyzer:
    def __init__(self, ner_model_path="./custom_ner_model", db_path="./chroma_db"):
        print("Initializing Analyzer...")
        self.nlp = spacy.load(ner_model_path)
        self.embedding_model = SentenceTransformer('BAAI/bge-base-en-v1.5')

        self.client = chromadb.PersistentClient(path=db_path)
        self.collection = self.client.get_collection(name="mooc_courses")

        # Threshold for deciding if a skill is a match (e.g., "AWS" and "Amazon Web Services")
        self.similarity_threshold = 0.85

    def extract_text_from_pdf(self, pdf_path):
        """Extract text from resume PDF without breaking multi-column layouts."""
        text = ""
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                text += page.extract_text() + "\n"
        return text

    def extract_skills(self, text):
        """Run custom NER to grab HARD_SKILL, SOFT_SKILL, TOOL."""
        doc = self.nlp(text)
        skills = list(set([ent.text.lower() for ent in doc.ents if ent.label_ in ["HARD_SKILL", "TOOL"]]))
        return skills

    def compute_semantic_delta(self, resume_skills, jd_skills):
        """Finds skills in JD that are missing from the Resume using embeddings."""
        if not jd_skills:
            return []
        if not resume_skills:
            return jd_skills

        # Embed all skills
        resume_embeddings = self.embedding_model.encode(resume_skills, normalize_embeddings=True)
        jd_embeddings = self.embedding_model.encode(jd_skills, normalize_embeddings=True)

        # Calculate cosine similarity matrix between JD skills and Resume skills
        sim_matrix = cosine_similarity(jd_embeddings, resume_embeddings)

        skill_gaps = []
        for idx, jd_skill in enumerate(jd_skills):
            # Find the best matching skill in the resume for this JD skill
            best_match_score = np.max(sim_matrix[idx])

            # If the best match is below our threshold, it's a gap
            if best_match_score < self.similarity_threshold:
                skill_gaps.append(jd_skill)

        return skill_gaps

    def generate_syllabus(self, skill_gaps, top_k=3):
        """Queries ChromaDB to find courses that teach the missing skills."""
        if not skill_gaps:
            return "No skill gaps detected. You are a perfect match!"

        query = "Represent this sentence for searching relevant passages: Courses teaching " + ", ".join(skill_gaps)

        query_embedding = self.embedding_model.encode([query], normalize_embeddings=True).tolist()

        results = self.collection.query(
            query_embeddings=query_embedding,
            n_results=top_k
        )

        return results['metadatas'][0], results['documents'][0]

    def run_analysis(self, resume_pdf_path, jd_text):
        """End-to-end pipeline execution."""
        print("Extracting text from resume...")
        resume_text = self.extract_text_from_pdf(resume_pdf_path)

        print("Extracting entities...")
        resume_skills = self.extract_skills(resume_text)
        jd_skills = self.extract_skills(jd_text)

        print(f"Resume Skills Found: {len(resume_skills)}")
        print(f"JD Skills Found: {len(jd_skills)}")

        print("Computing semantic delta...")
        gaps = self.compute_semantic_delta(resume_skills, jd_skills)
        print(f"Identified Gaps: {gaps}")

        print("Generating personalized syllabus...")
        courses = self.generate_syllabus(gaps)

        return {
            "resume_skills": resume_skills,
            "jd_skills": jd_skills,
            "skill_gaps": gaps,
            "recommended_courses": courses
        }

Writing skill_gap_analyzer.py


In [48]:
import argparse
import json
import os

# Importing the modules we built in the previous steps
# Ensure those files are named exactly as imported below
from skill_gap_analyzer import SkillGapAnalyzer

def run_setup():
    """Runs the one-time data engineering and model training pipelines."""
    print("=== STARTING SYSTEM SETUP ===")

    # 1. Prepare NER Training Data
    print("\n--- Step 1: Preparing NER Data ---")
    prep = NERDataPrep()
    training_data = prep.format_huggingface_data(num_samples=200)

    # 2. Train Custom NER Model
    print("\n--- Step 2: Training Custom spaCy Model ---")
    # In a real scenario, you'd save/load this from the JSON,
    # but we can pass the list directly for simplicity here.
    train_custom_ner(training_data, output_dir="./custom_ner_model", iterations=15)

    # 3. Build Vector Database
    print("\n--- Step 3: Populating ChromaDB ---")
    if not os.path.exists("mooc_dataset.csv"):
        print("Error: 'mooc_dataset.csv' not found. Please create it using the sample data provided earlier.")
        return

    db_builder = VectorDBBuilder(db_path="./chroma_db")
    db_builder.process_and_store("mooc_dataset.csv")

    print("\n=== SETUP COMPLETE ===")
    print("Your models and database are ready. You can now run the analyzer.")

def run_analysis(resume_path, jd_path):
    """Runs the production inference pipeline for a single user."""
    print("=== STARTING SKILL GAP ANALYSIS ===")

    # Validate files exist
    if not os.path.exists(resume_path) or not os.path.exists(jd_path):
        print(f"Error: Could not find {resume_path} or {jd_path}.")
        print("Please ensure your sample PDF and Job Description text files exist.")
        return

    # Load the Job Description Text
    with open(jd_path, 'r', encoding='utf-8') as file:
        jd_text = file.read()

    # Initialize the Analyzer (loads the custom model and DB we built in setup)
    analyzer = SkillGapAnalyzer(ner_model_path="./custom_ner_model", db_path="./chroma_db")

    # Run the core logic
    results = analyzer.run_analysis(resume_pdf_path=resume_path, jd_text=jd_text)

    # Output the final JSON-like payload
    print("\n=== FINAL RESULTS ===")
    print(f"Resume Skills: {results['resume_skills']}")
    print(f"Target JD Skills: {results['jd_skills']}")
    print(f"\nCRITICAL SKILL GAPS IDENTIFIED:\n -> {results['skill_gaps']}")

    if results['recommended_courses'] != "No skill gaps detected. You are a perfect match!":
        print("\nRECOMMENDED SYLLABUS TO BRIDGE THE GAP:")
        for idx, (meta, doc) in enumerate(zip(results['recommended_courses'][0], results['recommended_courses'][1])):
            print(f" {idx + 1}. {meta['title']} | Taught: {meta['clean_skills']}")
    else:
         print(results['recommended_courses'])

In [49]:
# 1. Run the setup once
run_setup()

# 2. Once setup is complete, run the analysis directly

=== STARTING SYSTEM SETUP ===

--- Step 1: Preparing NER Data ---
Formatting 200 samples for spaCy...

--- Step 2: Training Custom spaCy Model ---
Loading base spacy model...
Beginning training loop...
Iteration 1 - Losses: {'ner': np.float32(80.64702)}
Iteration 2 - Losses: {'ner': np.float32(43.5393)}
Iteration 3 - Losses: {'ner': np.float32(39.662094)}
Iteration 4 - Losses: {'ner': np.float32(36.26269)}
Iteration 5 - Losses: {'ner': np.float32(30.597935)}
Iteration 6 - Losses: {'ner': np.float32(19.756266)}
Iteration 7 - Losses: {'ner': np.float32(7.534987)}
Iteration 8 - Losses: {'ner': np.float32(6.408064)}
Iteration 9 - Losses: {'ner': np.float32(7.8457327)}
Iteration 10 - Losses: {'ner': np.float32(3.5463698)}
Iteration 11 - Losses: {'ner': np.float32(2.0939627)}
Iteration 12 - Losses: {'ner': np.float32(6.155659)}
Iteration 13 - Losses: {'ner': np.float32(1.0400813)}
Iteration 14 - Losses: {'ner': np.float32(1.9680141)}
Iteration 15 - Losses: {'ner': np.float32(1.1614771)}
Savi

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading data from mooc_dataset.csv...
Generating embeddings for 5 courses...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Storing in ChromaDB...
Vector database successfully populated.

=== SETUP COMPLETE ===
Your models and database are ready. You can now run the analyzer.


In [50]:
run_analysis(resume_path="SmithResume.pdf", jd_path="JD.txt")

=== STARTING SKILL GAP ANALYSIS ===
Initializing Analyzer...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracting text from resume...
Extracting entities...
Resume Skills Found: 7
JD Skills Found: 4
Computing semantic delta...
Identified Gaps: ['python', 'koramangala', 'interns']
Generating personalized syllabus...

=== FINAL RESULTS ===
Resume Skills: ['microsoft', 'canada', 'australia', 'us', 'sql', 'uk', 'michael']
Target JD Skills: ['python', 'koramangala', 'interns', 'sql']

CRITICAL SKILL GAPS IDENTIFIED:
 -> ['python', 'koramangala', 'interns']

RECOMMENDED SYLLABUS TO BRIDGE THE GAP:
 1.  Scikit-Learn | Taught: Numpy
 2.  Apache Spark | Taught: Python
 3.  HTML | Taught: Frontend Development
